# Segment 3: Dataset Images

In Segment 2 we asked: *"What synthetic image maximally activates a neuron?"*

Now we ask the complementary question:

> **"Which real images from ImageNet most strongly activate each neuron?"**

This grounds our understanding in reality. Synthetic activation maximization can produce abstract patterns, but seeing *actual* photos that fire a neuron tells us what it detects in practice.

**Task:** For each of the first 10 neurons of `mixed4a`, find the top 10 most-activating images from the ImageNet validation set (50k images, streamed from HuggingFace).

In [ ]:
# Cell 1: Setup
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

import matplotlib
matplotlib.use('Agg')

import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from IPython.display import display, Image as IPImage
import io
import heapq
from torchvision import transforms

def show_fig():
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', dpi=120)
    buf.seek(0)
    display(IPImage(data=buf.read()))
    plt.close()

# Patch lucent for PyTorch 2.x (H100 CUBLAS fix)
import lucent.optvis.param.color as color_module

def _patched_linear_decorrelate_color(tensor):
    batch, channels, height, width = tensor.shape
    t_flat = tensor.permute(0, 2, 3, 1).reshape(-1, channels)
    cc = np.array(color_module.color_correlation_normalized.T, dtype=np.float32)
    color_matrix = torch.from_numpy(cc).to(dtype=tensor.dtype, device=tensor.device)
    result = torch.mm(t_flat, color_matrix)
    return result.reshape(batch, height, width, channels).permute(0, 3, 1, 2)

color_module._linear_decorrelate_color = _patched_linear_decorrelate_color

from lucent.modelzoo import inceptionv1

device = torch.device("cuda:0")
print(f"Using: {torch.cuda.get_device_name(0)}")

model = inceptionv1(pretrained=True).to(device).eval()
print("InceptionV1 loaded")

In [ ]:
# Cell 2: Stream ImageNet validation set from HuggingFace
from huggingface_hub import login
from datasets import load_dataset

login(token=os.environ.get("HF_TOKEN_PERSONAL"))

imagenet_stream = load_dataset("ILSVRC/imagenet-1k", split="validation", streaming=True)
print("ImageNet validation set streaming ready (50,000 images)")

# ImageNet preprocessing
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## Finding Top-Activating Images

We stream every image from the ImageNet validation set through InceptionV1 and record the activation of each neuron in `mixed4a`. For each neuron, we keep the top-10 images with the highest mean spatial activation using a min-heap.

Since we're streaming, we also store a thumbnail of each top image (no need to re-download later).

**Parallelization:** Three things happen concurrently via a thread pool:
1. **Fetching** the next batch of raw samples from HuggingFace (network I/O)
2. **Preprocessing** the current batch on CPU (resize, crop, normalize)
3. **GPU inference** on the previous batch (batched forward pass, 64 images at once)

This pipeline keeps the GPU busy while the CPU/network prepare the next batch.

In [ ]:
# Cell 3: Stream all 50k images and track top activations (batched + prefetched)
LAYER_NAME = "mixed4a"
NUM_NEURONS = 10
TOP_K = 10
BATCH_SIZE = 64

# Hook to capture mixed4a activations
captured = {}

def hook_fn(module, input, output):
    captured[LAYER_NAME] = output.detach()

for name, module in model.named_modules():
    if name == LAYER_NAME:
        module.register_forward_hook(hook_fn)
        break

# Min-heaps: (activation_value, image_index, thumbnail, activation_map)
top_k_per_neuron = [[] for _ in range(NUM_NEURONS)]
counter = 0

def preprocess_batch(samples):
    """Preprocess a list of streamed samples into a batch tensor + thumbnails."""
    tensors = []
    thumbs = []
    for sample in samples:
        pil_img = sample["image"].convert("RGB")
        tensors.append(preprocess(pil_img))
        thumb = pil_img.copy()
        thumb.thumbnail((224, 224))
        thumbs.append(thumb)
    return torch.stack(tensors), thumbs

def collect_batch(stream_iter, n):
    """Collect up to n samples from the stream iterator."""
    batch = []
    for _ in range(n):
        try:
            batch.append(next(stream_iter))
        except StopIteration:
            break
    return batch

print(f"Streaming ImageNet validation set, tracking top-{TOP_K} per neuron...")
print(f"Batch size: {BATCH_SIZE} (GPU) + threaded prefetch")

from concurrent.futures import ThreadPoolExecutor
import time

stream_iter = iter(imagenet_stream)
t0 = time.time()

with torch.no_grad(), ThreadPoolExecutor(max_workers=2) as executor:
    # Prefetch first batch
    raw_batch = collect_batch(stream_iter, BATCH_SIZE)
    
    while raw_batch:
        # Submit preprocessing to background thread
        future = executor.submit(preprocess_batch, raw_batch)
        
        # Meanwhile, start fetching next raw batch from the stream
        next_raw_future = executor.submit(collect_batch, stream_iter, BATCH_SIZE)
        
        # Wait for preprocessing to finish
        batch_tensor, thumbs = future.result()
        batch_tensor = batch_tensor.to(device)
        
        # Forward pass (batched!)
        _ = model(batch_tensor)
        acts = captured[LAYER_NAME]  # [batch, channels, H, W]
        
        # Update heaps for each image in the batch
        for i in range(acts.shape[0]):
            for neuron in range(NUM_NEURONS):
                val = acts[i, neuron].mean().item()
                
                # Only compute/store act_map if this image might make the top-K
                if len(top_k_per_neuron[neuron]) < TOP_K or val > top_k_per_neuron[neuron][0][0]:
                    act_map = acts[i, neuron].cpu().numpy()
                    entry = (val, counter, thumbs[i], act_map)
                    
                    if len(top_k_per_neuron[neuron]) < TOP_K:
                        heapq.heappush(top_k_per_neuron[neuron], entry)
                    else:
                        heapq.heapreplace(top_k_per_neuron[neuron], entry)
            
            counter += 1
        
        if counter % 5000 < BATCH_SIZE:
            elapsed = time.time() - t0
            rate = counter / elapsed
            print(f"  {counter}/50000 images  ({rate:.0f} img/s, {elapsed:.0f}s elapsed)")
        
        # Get next raw batch (was fetching in background)
        raw_batch = next_raw_future.result()

# Sort descending by activation
top_k_per_neuron = [sorted(h, key=lambda x: -x[0]) for h in top_k_per_neuron]

elapsed = time.time() - t0
print(f"\nDone! Scanned {counter} images in {elapsed:.0f}s ({counter/elapsed:.0f} img/s)")

In [ ]:
# Cell 4: Display top-10 images for each neuron

for neuron in range(NUM_NEURONS):
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    fig.suptitle(f"{LAYER_NAME} : neuron {neuron}", fontsize=18, fontweight='bold')
    
    for rank, (ax, (act_val, _, thumb, _)) in enumerate(zip(axes.flat, top_k_per_neuron[neuron])):
        ax.imshow(thumb)
        ax.set_title(f"#{rank+1}  act={act_val:.2f}", fontsize=10)
        ax.axis('off')
    
    plt.tight_layout()
    show_fig()

## Activation Heatmaps

Beyond knowing *which* images activate a neuron, we can see *where* in the image the neuron fires. We overlay the spatial activation map on the top image for each neuron.

In [ ]:
# Cell 5: Activation heatmaps for the top image of each neuron

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("Where each neuron fires on its top image", fontsize=16, fontweight='bold')

for neuron, ax in enumerate(axes.flat):
    act_val, _, thumb, act_map = top_k_per_neuron[neuron][0]
    
    # Upsample activation map to thumbnail size
    w, h = thumb.size
    act_map_resized = np.array(Image.fromarray(act_map).resize((w, h), Image.BILINEAR))
    
    ax.imshow(thumb)
    ax.imshow(act_map_resized, cmap='hot', alpha=0.5)
    ax.set_title(f"neuron {neuron}  ({act_val:.2f})", fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
show_fig()

## Summary

**What we did:**
- Streamed all 50,000 ImageNet validation images through InceptionV1
- Recorded mean spatial activation for neurons 0-9 of `mixed4a`
- Displayed top-10 real images per neuron + activation heatmaps

**Why this matters:**
- Activation maximization (Segment 2) shows *what* a neuron looks for in theory
- Dataset images show *what* it responds to in practice
- Together they build confidence that we actually understand the neuron

This is exactly **Argument 2** from the Circuits paper: *"The ImageNet images that cause these neurons to strongly fire are reliably [the expected pattern] in the expected orientation."*

---

**Reference:** [Zoom In: An Introduction to Circuits](https://distill.pub/2020/circuits/zoom-in/)